In [1]:
from pathlib import Path

import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F
from torchvision.models import resnet18
import torchvision.transforms as transforms

from membership_dataset import MembershipDataset

In [2]:
# config
BASE = Path().resolve()
# BASE = Path(__file__).parent
PUB_PATH = BASE / "pub.pt"
PRIV_PATH = BASE / "priv.pt"
MODEL_PATH = BASE / "model.pt"
OUTPUT_CSV = BASE / "submission.csv"

In [3]:
model = resnet18(weights=None)
model.conv1 = torch.nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)  # small images
model.maxpool = torch.nn.Identity()   # remove maxpool
model.fc = torch.nn.Linear(512, 9)   # 9 classes
model.load_state_dict(torch.load("model.pt", map_location="cpu"))
model.eval()                          # critical for inference

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

In [4]:
MEAN = [0.7406, 0.5331, 0.7059]   # per-channel mean used during training
STD = [0.1491, 0.1864, 0.1301]    # per-channel std used during training

transform = transforms.Compose([
    transforms.Resize(32),                     # resize to match training resolution
    transforms.Normalize(mean=MEAN, std=STD),  # (pixel - mean) / std per channel
])

In [5]:
pub_ds = torch.load(PUB_PATH, weights_only=False)
priv_ds = torch.load(PRIV_PATH, weights_only=False)

pub_ds.transform = transform    # attach normalization after loading
priv_ds.transform = transform

# loader = DataLoader(pub_ds, batch_size=64, shuffle=False)
# loader = DataLoader(priv_ds, batch_size=64, shuffle=False)

In [6]:
def collate_fn(batch):
    ids, imgs, labels, _ = zip(*batch)
    return list(ids), torch.stack(list(imgs)), torch.tensor(labels)

loader = DataLoader(priv_ds, batch_size=64, shuffle=False, collate_fn=collate_fn)

In [ ]:
# all_ids = []
# all_scores = []
# all_labels = []
# all_preds = []

# with torch.no_grad():
#     for id_, imgs, labels in loader:
#         imgs = imgs.to(device)
#         logits = model(imgs)
#         probs = F.softmax(logits, dim=1)
#         scores = probs.max(dim=1).values.cpu().numpy()  # max prob as membership score
#         predicted_label = probs.argmax(dim=1).cpu().numpy() # predicted label

#         all_ids.extend(id_)
#         all_scores.extend(scores.tolist())
#         all_labels.extend(labels.tolist())
#         all_preds.extend(predicted_label.tolist())


In [ ]:
# df = pd.DataFrame({
#     "id": all_ids,
#     "label": all_labels,
#     "score": all_scores,
#     "predicted_label": all_preds,
# })

# df

,id,label,score,predicted_label
0,34566,3,0.961085,3
1,50272,5,0.946675,5
2,69232,5,0.948389,5
3,86223,5,0.603356,5
4,53524,5,0.949084,5
...,...,...,...,...
13995,6232,0,0.950047,0
13996,47970,0,0.956411,0
13997,55360,4,0.957415,4
13998,42541,6,0.960936,6


In [ ]:
# df.to_csv("max_confidence_test_priv_data.csv", index=False)

In [10]:
df = pd.read_csv("max_confidence_test_priv_data.csv")
df

,id,label,score,predicted_label
0,34566,3,0.961085,3
1,50272,5,0.946675,5
2,69232,5,0.948389,5
3,86223,5,0.603356,5
4,53524,5,0.949084,5
...,...,...,...,...
13995,6232,0,0.950047,0
13996,47970,0,0.956411,0
13997,55360,4,0.957415,4
13998,42541,6,0.960936,6


In [11]:
df["label"].value_counts()

label
8    2117
5    1836
2    1574
3    1541
0    1520
1    1483
7    1431
4    1261
6    1237
Name: count, dtype: int64

In [12]:
df["predicted_label"].value_counts()

predicted_label
8    2128
5    1829
2    1577
3    1537
0    1520
1    1485
7    1426
4    1273
6    1225
Name: count, dtype: int64

In [13]:
df["score"].describe()

count    14000.000000
mean         0.946315
std          0.045349
min          0.243604
25%          0.946086
50%          0.953159
75%          0.959964
max          0.991892
Name: score, dtype: float64

In [14]:
# OBSERVATION:
# Both members and non-members have high confidence scores. The true labels are almost uniformly distributed. 
# Hence using just the confidence score is a very poor indicator of membership.